In [1]:
import utils
import keras
import numpy as np
from aim.keras import AimCallback

2026-03-02 16:33:44.221380: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 16:33:45.349560: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 16:33:57.183598: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
(train_x, train_y), (val_x, val_y), (test_x, test_y) = utils.read_stratified_data(columns=('lighting', 'model'))

/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/275815406_Andrew-Crowley-1_trans_NvBQzQNjv4BqJgZjG4XE8BZGTSy9SLp5TPzOL_bEerq8T.jpg
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/kisspng-2018-tesla-model-s-tesla-motors-car-ele_fLj30kA.2230115115224762368998.png
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/Edmu_EmY1CYe.-2020-Tesla-Model-Ywire-38540856-1611748943-973_634x3962-1280x720.jpg
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/black-tesla-model-x-carbon-fiber-spoiler-mx22-forged-aftermarket-wheel_tCrLNla.jpg


In [3]:
print(train_x.shape)
print(train_y.shape)
print(val_x.shape)
print(val_y.shape)
print(test_x.shape)
print(test_y.shape)

(3036, 300, 300, 3)
(3036, 6)
(650, 300, 300, 3)
(650, 6)
(649, 300, 300, 3)
(649, 6)


In [4]:
input = keras.Input(shape=(train_x[0].shape))

x = keras.layers.Conv2D(filters=256, kernel_size=3, activation="relu")(input)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Flatten()(x)

gate1 = keras.layers.Dense(1, activation="sigmoid", name="gate1")(x)
gate2 = keras.layers.Dense(8, activation="softmax", name="gate2")(x)

model = keras.Model(inputs=input, outputs={"gate1": gate1, "gate2": gate2}, name="Overfitted_model")
model.summary()

opt = keras.optimizers.Adam(learning_rate=6e-4)

model.compile(
    optimizer=opt,
    loss={
        "gate1": keras.losses.BinaryCrossentropy(),
        "gate2": keras.losses.SparseCategoricalCrossentropy(),
    },
    metrics={
        "gate1": [keras.metrics.BinaryAccuracy(name="acc")],
    },
    weighted_metrics={
        "gate2": [keras.metrics.SparseCategoricalAccuracy(name="acc")],
    },
)


I0000 00:00:1772465686.804421   21915 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9569 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "Overfitted_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 298, 298,  │      7,168 │ input_layer[0][0] │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 149, 149,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │    295,040 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 73, 73,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 71, 71,    │     73,792 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 35, 35,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 78400)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gate1 (Dense)       │ (None, 1)         │     78,401 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gate2 (Dense)       │ (None, 8)         │    627,208 │ flatten[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,081,609 (4.13 MB)

 Trainable params: 1,081,609 (4.13 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
aim_cb = AimCallback(
    repo=".",                      
    experiment="Vehicle_Gate_Test",
)

# Try to save hparams to Aim using a small helper (aim_utils.save_hparams)
try:
    from aim_utils import save_hparams
except Exception:
    import sys, os
    sys.path.append(os.path.join(os.getcwd(), 'notebooks'))
    from aim_utils import save_hparams

hparams = {
    "learning_rate": opt.learning_rate,
    "batch_size": 8,
    "epochs": 5,
    "optimizer": "Adam",
    "model": "Overfitted_model",
}
save_hparams(hparams, repo='.', experiment='Vehicle_Gate_Test')

sample_weight={
    "gate1": np.ones(len(train_y)),
    "gate2": 1 - train_y['gate1'] 
}

validation_data=(val_x, {'gate1': val_y["gate1"], 'gate2': val_y["gate2"]})

y = {'gate1': train_y["gate1"], 'gate2': train_y["gate2"]}
history = model.fit(epochs=5, batch_size=8, x=train_x, y=y, sample_weight=sample_weight, validation_data=validation_data, callbacks=[aim_cb])

Epoch 1/5
380/380 ━━━━━━━━━━━━━━━━━━━━ 102s 267ms/step - gate1_acc: 0.9977 - gate1_loss: 0.0136 - gate2_acc: 0.9962 - gate2_loss: 0.0325 - loss: 0.0462 - val_gate1_acc: 0.7477 - val_gate1_loss: 0.8568 - val_gate2_acc: 0.2123 - val_gate2_loss: 6.4625 - val_loss: 7.3329
Epoch 2/5
380/380 ━━━━━━━━━━━━━━━━━━━━ 125s 329ms/step - gate1_acc: 0.9990 - gate1_loss: 0.0126 - gate2_acc: 0.9956 - gate2_loss: 0.0350 - loss: 0.0476 - val_gate1_acc: 0.7677 - val_gate1_loss: 0.9125 - val_gate2_acc: 0.1938 - val_gate2_loss: 6.8745 - val_loss: 7.7744
Epoch 3/5
380/380 ━━━━━━━━━━━━━━━━━━━━ 108s 286ms/step - gate1_acc: 1.0000 - gate1_loss: 0.0066 - gate2_acc: 0.9969 - gate2_loss: 0.0172 - loss: 0.0238 - val_gate1_acc: 0.7600 - val_gate1_loss: 1.1606 - val_gate2_acc: 0.2062 - val_gate2_loss: 8.8308 - val_loss: 10.0051
Epoch 4/5
380/380 ━━━━━━━━━━━━━━━━━━━━ 106s 278ms/step - gate1_acc: 1.0000 - gate1_loss: 0.0038 - gate2_acc: 0.9975 - gate2_loss: 0.0155 - loss: 0.0193 - val_gate1_acc: 0.7600 - val_gate1_loss